# kl-divergence-gaussian-closed-form — ex1: closed-form Gaussian KL with per-sample bar chart

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kl-divergence-gaussian-closed-form`. Running the final beacon cell reports progress against the `VAE: KL divergence Gaussian closed-form` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: KL divergence Gaussian closed-form` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kl-divergence-gaussian-closed-form`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kl-divergence-gaussian-closed-form"
DD_SUBTOPIC = "VAE: KL divergence Gaussian closed-form"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## KL divergence (Gaussian, closed-form) — quick refresher

For a Gaussian posterior `q(z|x) = N(mu, sigma^2)` vs the standard-normal prior `p(z) = N(0, 1)`, the KL divergence has a closed form:

```
KL(q || p) = -0.5 * sum(1 + logsigma - mu^2 - exp(logsigma))
```

where `logsigma` is the encoder's log-VARIANCE output (yes — the ARENA convention uses `logsigma` as the variable name even though it is treating it as log-var; the formula below matches that).

**Sum over latent dim, mean over batch.** Sum across `latent_dim` to get per-sample KL `(B,)`. Mean across batch (or sum, scaled later) to get the scalar loss term:
```python
kl_per_sample = -0.5 * (1 + logsigma - mu.pow(2) - logsigma.exp()).sum(dim=-1)
kl_scalar = kl_per_sample.mean()
```

**Sanity check.** When `mu=0` and `logsigma=0` (i.e. `sigma=1`), the posterior == prior and `KL == 0`. Easy unit test: build a zero tensor, run your function, assert near zero.

### Exercise 1 — closed-form Gaussian KL with per-sample bar chart

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the closed-form Gaussian-vs-standard-normal KL `-0.5 * sum(1 + logsigma - mu^2 - exp(logsigma))` to compute (a) per-sample KL of shape `(B,)` and (b) the batch-mean scalar.
> Keywords: kl-divergence, vae, closed-form, gaussian
> ```

**KCs targeted:** `kl-gaussian-formula`, `kl-sum-then-mean`

Implement `ex1_kl_gaussian(mu, logsigma)`. The closed-form KL term that completes the VAE ELBO:

1. `mu` and `logsigma` both have shape `(B, latent_dim)` — the encoder's Gaussian-parameter outputs.
2. Compute the per-element KL contribution:
   `-0.5 * (1 + logsigma - mu**2 - exp(logsigma))` → `(B, latent_dim)`.
3. Sum across `latent_dim` (the last axis) → per-sample KL `(B,)`.
4. Mean across batch → scalar (0-D tensor).
5. Return the tuple `(per_sample_kl, scalar_kl)`.

Input: `mu`, `logsigma` — `(B, latent_dim)` float tensors.
Output: tuple `((B,) tensor, scalar tensor)`.

The visualization bar-charts the per-sample KL on a synthetic batch where some samples have `mu` and `logsigma` near zero (low KL, posterior ≈ prior) and others are far from zero (high KL).

In [ ]:
def ex1_kl_gaussian(mu: Tensor, logsigma: Tensor) -> tuple[Tensor, Tensor]:
    """Return (per-sample KL of shape (B,), scalar batch mean)."""
    raise NotImplementedError()


def _test_ex1():
    import math

    # Posterior == prior case: mu=0, logsigma=0 → KL = 0.
    B, latent_dim = 4, 5
    mu_z = t.zeros(B, latent_dim)
    ls_z = t.zeros(B, latent_dim)
    per_sample, scalar = ex1_kl_gaussian(mu_z, ls_z)
    assert per_sample.shape == (B,), f'expected (B,), got {tuple(per_sample.shape)}'
    assert scalar.shape == (), f'expected scalar (), got {tuple(scalar.shape)}'
    assert t.allclose(per_sample, t.zeros(B), atol=1e-6), f'KL must be 0 at posterior=prior, got {per_sample}'
    assert t.allclose(scalar, t.tensor(0.0), atol=1e-6)

    # Closed-form check vs explicit per-dim computation.
    rng = t.Generator().manual_seed(0)
    mu = t.randn(3, 4, generator=rng)
    ls = t.randn(3, 4, generator=rng)
    expected_per_elem = -0.5 * (1 + ls - mu.pow(2) - ls.exp())
    expected_per_sample = expected_per_elem.sum(dim=-1)
    expected_scalar = expected_per_sample.mean()
    ps, sc = ex1_kl_gaussian(mu, ls)
    assert t.allclose(ps, expected_per_sample, atol=1e-5), 'per-sample KL formula mismatch'
    assert t.allclose(sc, expected_scalar, atol=1e-5), 'scalar KL must equal mean of per-sample'

    # Variance-only deviation: mu=0, logsigma != 0.
    # When mu=0 and logsigma=c, per-sample KL = latent_dim * (-0.5 * (1 + c - exp(c))).
    c = 0.5
    mu_var = t.zeros(2, 6)
    ls_var = t.full((2, 6), c)
    ps_var, _ = ex1_kl_gaussian(mu_var, ls_var)
    expected_kl = 6 * (-0.5 * (1 + c - math.exp(c)))
    assert t.allclose(ps_var, t.full((2,), expected_kl), atol=1e-5), (
        f'variance-only KL mismatch: got {ps_var}, expected {expected_kl}'
    )

    # KL must be non-negative (mathematical fact).
    for _ in range(5):
        test_mu = t.randn(5, 8)
        test_ls = t.randn(5, 8)
        p, _ = ex1_kl_gaussian(test_mu, test_ls)
        assert (p >= -1e-5).all(), f'KL must be >= 0, got {p}'

    # --- Visualization: per-sample KL bar chart on a graded batch ---
    B_viz = 12
    # Samples 0..3 near prior; 4..7 medium drift; 8..11 large drift.
    drift_levels = t.cat([
        t.full((4,), 0.0),
        t.full((4,), 1.0),
        t.full((4,), 3.0),
    ])
    mu_viz = drift_levels.unsqueeze(-1).expand(B_viz, 6)
    ls_viz = (0.3 * drift_levels).unsqueeze(-1).expand(B_viz, 6)
    per_viz, _ = ex1_kl_gaussian(mu_viz, ls_viz)
    colors = ['steelblue'] * 4 + ['orange'] * 4 + ['crimson'] * 4
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(range(B_viz), per_viz.numpy(), color=colors, edgecolor='black')
    ax.set_xlabel('batch sample idx')
    ax.set_ylabel('per-sample KL')
    ax.set_title('ex1 per-sample KL — blue: posterior≈prior, orange: drift, red: far')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_kl_gaussian(mu: Tensor, logsigma: Tensor) -> tuple[Tensor, Tensor]:
    per_elem = -0.5 * (1 + logsigma - mu.pow(2) - logsigma.exp())
    per_sample = per_elem.sum(dim=-1)
    scalar = per_sample.mean()
    return per_sample, scalar
```

**Where the closed form comes from.** For two Gaussians `q = N(mu, sigma^2)` and `p = N(0, 1)`, KL has a closed form:
`KL(q || p) = 0.5 * (mu^2 + sigma^2 - 1 - log(sigma^2))`.
Substituting `log(sigma^2) = logsigma` (ARENA convention treats `logsigma` as log-variance) and rearranging gives the formula you implemented.

**Sum across latent, mean across batch.** Per-sample KL is the sum over latent dims because dims are INDEPENDENT under the diagonal-covariance assumption. Mean across batch makes the loss comparable across batch sizes.

**KL is always ≥ 0.** A solid sanity test for your implementation. If you ever get negative KL, you've sign-flipped the `-0.5` or dropped a term.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()